In [42]:
from akula import API_TOKEN
#print(API_TOKEN)

ModuleNotFoundError: No module named 'akula'

In [10]:
import telebot
from telebot import types
import sqlite3
from datetime import datetime
import random

user_data = {}
movie_id = None
bot = telebot.TeleBot(API_TOKEN)

from datetime import datetime



def calculate_age(birth_date_str):
    birth_date = datetime.strptime(birth_date_str, '%Y-%m-%d').date()
    today = datetime.today()
    age = today.year - birth_date.year - ((today.month, today.day) < (birth_date.month, birth_date.day))
    return age



# Проверка, существует ли пользователь в базе данных
def user_exists(user_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''SELECT COUNT(1) FROM users WHERE user_id = ?''', (user_id,))
    exists = cursor.fetchone()[0] > 0
    conn.close()
    return exists

# Сохранение информации о пользователе в базу данных
def save_user_info(user_info):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    if user_exists(user_info['user_id']):
        cursor.execute('''UPDATE users
                          SET first_name = ?, last_name = ?, username = ?, language_code = ?, is_bot = ?, birth_date = ?, last_activity_date = ?
                          WHERE user_id = ?''',
                       (user_info['first_name'], user_info.get('last_name'), user_info.get('username'),
                        user_info.get('language_code'), user_info['is_bot'], user_info.get('birth_date'),
                        user_info['last_activity_date'], user_info['user_id']))
    else:
        cursor.execute('''INSERT INTO users
                          (user_id, first_name, last_name, username, language_code, is_bot, birth_date, registration_date, last_activity_date)
                          VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)''',
                       (user_info['user_id'], user_info['first_name'], user_info.get('last_name'), user_info.get('username'),
                        user_info.get('language_code'), user_info['is_bot'], user_info.get('birth_date'),
                        user_info['registration_date'], user_info['last_activity_date']))
    conn.commit()
    conn.close()

# Обновление даты последней активности пользователя
def update_last_activity(user_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''UPDATE users 
                      SET last_activity_date = ? 
                      WHERE user_id = ?''',
                   (datetime.now().strftime('%Y-%m-%d %H:%M:%S'), user_id))
    conn.commit()
    conn.close()

def get_random_movie(user_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Получаем список ID фильмов, которые уже были оценены пользователем
    cursor.execute('''SELECT movie_id FROM actions WHERE user_id = ?''', (user_id,))
    rated_movies = [row[0] for row in cursor.fetchall()]

    # Получаем возраст пользователя
    cursor.execute('''SELECT birth_date FROM users WHERE user_id = ?''', (user_id,))
    birth_date = cursor.fetchone()[0]
    user_age = calculate_age(birth_date)

    # Получаем список всех фильмов, доступных для просмотра пользователем в зависимости от его возраста
    cursor.execute('''SELECT id FROM movies WHERE age_rating <= ?''', (user_age,))
    all_movies = [row[0] for row in cursor.fetchall()]

    # Проверяем, есть ли еще неоцененные фильмы
    unrated_movies = set(all_movies) - set(rated_movies)
    if not unrated_movies:
        # Если все фильмы уже оценены, возвращаем None
        return None

    # Получаем случайный фильм, который еще не был оценен пользователем
    movie_id = random.choice(list(unrated_movies))
    cursor.execute('''SELECT id, name, slogan, description, year
                      FROM movies
                      WHERE id = ?''', (movie_id,))
    movie = cursor.fetchone()
    conn.close()
    return movie



# Получение URL-адреса превью по ID фильма
def get_preview_url(movie_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''SELECT preview_url FROM posters WHERE movie_id = ?''', (movie_id,))
    preview_url = cursor.fetchone()
    conn.close()
    return preview_url[0] if preview_url else None


# Обработчик команды /start
@bot.message_handler(commands=['start'])
def send_welcome(message):
    user = message.from_user
    user_info = {
        'user_id': user.id,
        'first_name': user.first_name,
        'last_name': user.last_name,
        'username': user.username,
        'language_code': user.language_code,
        'is_bot': user.is_bot,
        'birth_date': None,
        'registration_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'last_activity_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    if user_exists(user.id):
        # Если пользователь уже существует, обновляем только дату последней активности
        update_last_activity(user.id)
        # Проверяем наличие даты рождения у пользователя
        if not user_has_birth_date(user.id):
            # Если дата рождения отсутствует, запрашиваем ее у пользователя
            bot.reply_to(message, "Привет! Пожалуйста, укажи свою дату рождения в формате ДД.ММ.ГГГГ.")
            bot.register_next_step_handler(message, process_birth_date)
        else:
            # Если дата рождения уже указана, продолжаем работу бота
            bot.reply_to(message, f"Привет, {user.first_name}! Добро пожаловать в наш бот для оценки фильмов.")
            send_random_movie(message)
    else:
        # Если пользователь новый, сохраняем всю информацию
        save_user_info(user_info)
        # Запрашиваем дату рождения у пользователя
        bot.reply_to(message, "Привет! Пожалуйста, укажи свою дату рождения в формате ДД.ММ.ГГГГ.")
        bot.register_next_step_handler(message, process_birth_date)

#Проверки наличия даты рождения у пользователя в базе данных
def user_has_birth_date(user_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''SELECT birth_date FROM users WHERE user_id = ?''', (user_id,))
    birth_date = cursor.fetchone()[0]
    conn.close()
    return birth_date is not None

#Обработка даты рождения пользователя
def process_birth_date(message):
    user = message.from_user
    birth_date_str = message.text
    try:
        birth_date = datetime.strptime(birth_date_str, '%d.%m.%Y').date()
        # Сохраняем дату рождения пользователя в базу данных
        conn = sqlite3.connect('movies.db')
        cursor = conn.cursor()
        cursor.execute('''UPDATE users SET birth_date = ? WHERE user_id = ?''', (birth_date, user.id))
        conn.commit()
        conn.close()
        # Продолжаем работу бота
        bot.reply_to(message, f"Спасибо! Теперь вы можете оценивать фильмы.")
        send_random_movie(message)
    except ValueError:
        # Если дата рождения введена некорректно, запрашиваем ее снова
        bot.reply_to(message, "Некорректный формат даты рождения. Пожалуйста, укажите дату рождения в формате ДД.ММ.ГГГГ.")
        bot.register_next_step_handler(message, process_birth_date)


# Отправка случайного фильма для оценки
def send_random_movie(message):
    global movie_id
    user = message.from_user
    movie = get_random_movie(user.id)
    # Создание кнопок для оценки фильма
    reply_markup = types.ReplyKeyboardMarkup(row_width=3, resize_keyboard=True)
    btn_dislike = types.KeyboardButton('👎')
    btn_menu = types.KeyboardButton('📺')
    btn_like = types.KeyboardButton('👍')
    reply_markup.add(btn_dislike, btn_menu, btn_like)
    
    if movie:
        movie_id, title, tagline, description, release_year = movie

        preview_url = get_preview_url(movie_id)



        # Формирование текста сообщения
        movie_text = f"*{title}* \n\n"
        if tagline:
            movie_text += f"*{tagline}*\n\n"
        if description:
            movie_text += f"{description}\n\n"
        movie_text += f"*Год фильма: {release_year}*"

        if preview_url:
            # Отправка фотографии с inline_markup
            bot.send_photo(message.chat.id, preview_url, caption=movie_text, parse_mode='Markdown', reply_markup=reply_markup)
        else:
            bot.send_message(message.chat.id, "Не удалось найти превью для фильма.")
    else:
        
        # Создаем клавиатуру с кнопкой "Найти в базе"
        markup = types.ReplyKeyboardMarkup(row_width=1, resize_keyboard=True)
        btn_search_db = types.KeyboardButton('Найти в базе данных другой')
        markup.add(btn_search_db)
    
        bot.reply_to(message, "Вы оценили все фильмы. Вы можете найти фильм по названию.", reply_markup=markup)




#Обработка '👎', '👍'
@bot.message_handler(func=lambda message: message.text in ['👎', '👍'])
def movie_rating_handler(message):
    user = message.from_user
    update_last_activity(user.id)  # Обновляем дату последней активности

    rating = None
    want_to_watch = None

    if message.text == '👎':
        want_to_watch = -1
        bot.reply_to(message, "Вы поставили отрицательную оценку.")
    elif message.text == '👍':
        want_to_watch = 1
        bot.reply_to(message, "Вы поставили положительную оценку.")

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    cursor.execute('''INSERT INTO actions
                      (user_id, movie_id, want_to_watch, rating)
                      VALUES (?, ?, ?, ?)''',
                   (user.id, movie_id, want_to_watch, rating))

    conn.commit()
    conn.close()

    # Отправляем следующий фильм после оценки
    send_random_movie(message)


#Обработка '📺'
@bot.message_handler(func=lambda message: message.text == '📺')
def show_liked_movies(message):
    global is_selecting_movie
    user = message.from_user
    update_last_activity(user.id)  # Обновляем дату последней активности

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Получаем список фильмов, на которые пользователь поставил "👍"
    cursor.execute('''SELECT movies.name, movies.year
                      FROM movies
                      JOIN actions ON movies.id = actions.movie_id
                      WHERE actions.user_id = ? AND actions.want_to_watch = 1''',
                   (user.id,))

    liked_movies = cursor.fetchall()
    conn.close()

    if liked_movies:
        # Создаем клавиатуру с кнопками "Изменить список" и "Найти в базе"
        markup = types.ReplyKeyboardMarkup(row_width=3, resize_keyboard=True)
        btn_edit_list = types.KeyboardButton('Изменить список')
        btn_search_db = types.KeyboardButton('Найти в базе данных другой')
        back_button = types.KeyboardButton("Передумал")
        markup.add(btn_edit_list, btn_search_db,back_button)
        
        # Формируем список фильмов в виде строки
        movies_list = "\n".join([f" `{title}` ({year})" for title, year in liked_movies])
        bot.reply_to(message, f"Список фильмов, на которые вы поставили 👍 :\n\n{movies_list}\n\nПожалуйста, введите название фильма, чтобы узнать площадки для просмотра.", reply_markup=markup, parse_mode="Markdown")
        is_selecting_movie = True
        bot.register_next_step_handler(message, show_watchability)
        
    else:
        
        bot.reply_to(message, "Вы еще не поставили 👍 ни одному фильму. Вы можете найти фильм в базе данных, чтобы добавить его в свой список.")
        send_random_movie(message)
        


#Удаляем фильм из списка
def edit_list(message):
    user_id = message.from_user.id
    
    if message.text.lower() == 'все':
        # Удаляем все фильмы из списка пользователя
        conn = sqlite3.connect('movies.db')
        cursor = conn.cursor()

        cursor.execute('''UPDATE actions
                          SET want_to_watch = 0
                          WHERE user_id = ?''', (user_id,))

        conn.commit()
        conn.close()

        bot.reply_to(message, "Ваш список фильмов успешно очищен.")
    else:
        
        movie_names = message.text.split(',')
    
    
        conn = sqlite3.connect('movies.db')
        cursor = conn.cursor()
    
        for movie_name in movie_names:
            # Удаляем пробелы в начале и конце названия фильма
            movie_name = movie_name.strip()
    
            # Обновляем значение want_to_watch на 0 для соответствующих фильмов
            cursor.execute('''UPDATE actions
                              SET want_to_watch = 0
                              WHERE user_id = ? AND movie_id IN (
                                  SELECT id FROM movies WHERE name = ?
                              )''', (user_id, movie_name))
    
        conn.commit()
        conn.close()
    
        bot.reply_to(message, f"Фильмы '{', '.join(movie_names)}' успешно удалены из вашего списка.")
    

    show_liked_movies(message)

#Осуществляем поиск фильма по имини
def search_in_database(message):
    user_id = message.from_user.id
    
        
    movie_name = message.text

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Ищем фильм в базе данных по полям 'name' и 'alternative_name'
    cursor.execute('''SELECT id FROM movies WHERE name = ? OR alternative_name = ?''', (movie_name, movie_name))
    movie_id = cursor.fetchone()

    if movie_id:
        movie_id = movie_id[0]

        # Добавляем информацию о действии пользователя в таблицу 'actions'
        cursor.execute('''INSERT INTO actions (user_id, movie_id, want_to_watch) VALUES (?, ?, 1)''', (user_id, movie_id))
        conn.commit()

        bot.reply_to(message, f"Фильм '{movie_name}' успешно добавлен в ваш список.")
        show_liked_movies(message)
    else:
        bot.reply_to(message, f"К сожалению, фильм '{movie_name}' не найден в базе данных.Напишите новое название фильма",search_in_database(message))
    conn.close()



def show_watchability(message):
    movie_name = message.text

    if message.text == 'Изменить список':
        bot.reply_to(message, "Пожалуйста, введите название фильма или фильмов, которые вы хотите удалить из списка.Если хоитите удалить несколько укажите через ',' или 'все'")
        # Регистрируем обработчик для следующего сообщения пользователя
        bot.register_next_step_handler(message, edit_list)
        return  
        
    elif message.text == 'Найти в базе данных другой':
        bot.reply_to(message, "Пожалуйста, введите название фильма для поиска в базе данных.")
        bot.register_next_step_handler(message, search_in_database)
        
        return
    else:
        print('Другое')
        
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Получаем информацию о фильме по названию
    cursor.execute('''SELECT id, name, year
                      FROM movies
                      WHERE name LIKE ?''',
                   (f"%{movie_name}%",))

    movie = cursor.fetchone()

    if movie:
        movie_id, title, year = movie

        # Получаем информацию о площадке для просмотра фильма
        cursor.execute('''SELECT service_name, link
                          FROM watchability
                          WHERE movie_id = ?''',
                       (movie_id,))

        watchability = cursor.fetchall()

        if watchability:
            # Format the platforms list as a string
            platforms_list = "\n".join([f"`{service_name[0]}`" for service_name in watchability])
    
            # Создаем клавиатуру с одной кнопкой "Передумал"
            keyboard = types.ReplyKeyboardMarkup(resize_keyboard=True)
            back_button = types.KeyboardButton("Передумал")
            keyboard.add(back_button)
    
            # Send the platforms list to the user with the keyboard
            bot.reply_to(message, f"Площадки для просмотра фильма '{title}' ({year}):\n\n{platforms_list}\n\nПожалуйста, выберите площадку для просмотра.", reply_markup=keyboard, parse_mode="Markdown")
    
            # Save the watchability list in the user's context
            user_data[message.from_user.id] = watchability
    
            # Register the platform selection handler
            bot.register_next_step_handler(message, handle_platform_selection)
        else:
            bot.reply_to(message, f"К сожалению, не удалось найти площадки для просмотра фильма '{title}' ({year}). Пожалуйста, выберите другой фильм.")
            show_liked_movies(message)
    else:
        # Создаем клавиатуру с кнопками "Выбрать другой фильм" и "Поискать новый фильм"
        markup = types.ReplyKeyboardMarkup(row_width=1, resize_keyboard=True)
        btn_search_new = types.KeyboardButton('Найти в базе данных другой')
        markup.add(btn_search_new)

        if movie_name == "Передумал":
            send_random_movie(message)
            return
            
        bot.reply_to(message, f"К сожалению, не удалось найти фильм с названием '{movie_name}' из списка фильмов на которые вы поставили 👍", reply_markup=markup)
        bot.register_next_step_handler(message, handle_movie_selection_buttons)


    conn.close()

    
def handle_platform_selection(message):
    user_id = message.from_user.id
    platform_name = message.text

    # Check if the user typed "Back" to go back to the previous step
    if platform_name == "Передумал":
        # Display the platform selection step again
        show_liked_movies(message)
        return

    # Check if the user's message is a valid platform name
    if platform_name in [platform[0] for platform in user_data[user_id]]:
        # Retrieve the corresponding link from the watchability list
        link = [platform[1] for platform in user_data[user_id] if platform[0] == platform_name][0]

        # Send the link to the user
        bot.reply_to(message, f"Ссылка на площадку для просмотра фильма: {link}")

        # Display a message that instructs the user to type "Back" to go back to the previous step
        bot.send_message(message.chat.id, "Чтобы вернуться обратно напишите /start")
    else:
        # Prompt the user to enter a valid platform name or type "Back" to go back to the previous step
        bot.reply_to(message, "Некорректный ввод. Пожалуйста, выберите площадку для просмотра из списка или напишите /start для выбора другого фильма.")

        # Register the platform selection handler for the next step
        bot.register_next_step_handler(message, handle_platform_selection)



#Обрабочик /drop
@bot.message_handler(commands=['drop'])
def drop_user_data(message):
    user = message.from_user

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Удаляем все записи пользователя из таблицы "actions"
    cursor.execute('''DELETE FROM actions WHERE user_id = ?''', (user.id,))

    conn.commit()
    conn.close()

    bot.reply_to(message, "Ваши сохраненные списки оценок успешно очищены.")
    send_random_movie(message)
    
# Обработчик некорректного ввода
@bot.message_handler(func=lambda message: message.text not in ['👎', '📺', '👍'])
def handle_incorrect_input(message):
    bot.reply_to(message, "Некорректный ввод. Пожалуйста, используйте кнопки для оценки фильма.")
    if message.text == 'Найти в базе данных другой':
        #print('Найти в базе данных другой')
        show_watchability(message)


def main():
    offset = None
    while True:
        updates = bot.get_updates(offset=offset, timeout=6000000)
        if updates:
            for update in updates:
                offset = update.update_id + 1
                process_update(update)
        else:
            time.sleep(1)
def process_update(update):
    if update.message:
        handle_message(update.message)
    elif update.callback_query:
        handle_callback_query(update.callback_query)
    # Добавьте обработку других типов обновлений по необходимости


if __name__ == '__main__':

    bot.polling(none_stop=True)
    